# 6 · Extract enhancer redundancy (Spark, cluster)

For each directed comparison (c1 → c2): find the nearest **active** enhancer per gene in each cell line, recompute the c1-nearest enhancer's **3D distance in c2's model**, and flag **redundancy** = nearest enhancer changed AND old enhancer is `large` (far) in c2 AND new enhancer is `small` (close) in c2.

Writes one parquet per comparison to `s3a://database/enhancer_redundancy/`. Analysis + plots happen locally in notebook 7. See `docs/superpowers/specs/2026-06-16-enhancer-redundancy-design.md`.

In [ ]:
%%configure -f
{"executorMemory": "12G", "executorCores": 12, "ttl": "12h", "heartbeatTimeoutInSecond": 43200, "numExecutors": 3}

In [ ]:
import json
import numpy as np
import pandas as pd
import s3fs
from pyspark.sql import Window
import pyspark.sql.functions as F

# --- constants --------------------------------------------------------------

# Broad "active" ChromHMM state set (matches notebook 1).
ACTIVE_STATES = ['TssA', 'TssAFlnk', 'TxFlnk', 'Tx', 'TxWk',
                 'EnhG', 'EnhG1', 'EnhG2', 'Enh', 'EnhA1', 'EnhA2']

# Per-cell-line result query IDs (same materialisations as notebook 1 — must
# retain enh/gene coords; confirm before running). cell_line -> query_id.
QUERY_IDS = {
    "GM12878": "a1fc46a9-93f8-424f-b41d-37bfd85d3b94",
    "H1ESC":   "f7bc6dac-6aa3-49e6-a2e5-c2ff27824c81",
    "HFFC6":   "f648f805-c3a9-4cf4-a108-94e6f5fa96c1",
}
USED_PROJECTS = ['whole_all_vs_all_gm12878_fix',
                 'whole_all_vs_all_h1esc_fix',
                 'whole_all_vs_all_hffc6_fix']
MODEL_REPOSITORY_BUCKET = "model-repository"   # confirm bucket / fs config

# Directed comparisons (3 unordered pairs -> 6 directed).
COMPARISONS = [("GM12878", "H1ESC"), ("H1ESC", "GM12878"),
               ("H1ESC", "HFFC6"), ("HFFC6", "H1ESC"),
               ("GM12878", "HFFC6"), ("HFFC6", "GM12878")]

# --- helpers ----------------------------------------------------------------

def chrom_tertile_thresholds(df, dist_col, chrom_col):
    # per-chromosome (q33, q67) of dist_col (matches notebook 3 tertiles)
    out = {}
    for chrom, grp in df.groupby(chrom_col):
        out[chrom] = (float(grp[dist_col].quantile(0.33)),
                      float(grp[dist_col].quantile(0.67)))
    return out

def proximity_categories(dist_series, chrom_series, thresholds):
    # small/mid/large vs per-chromosome thresholds; NaN or unknown chrom -> large
    cats = []
    for val, ch in zip(dist_series, chrom_series):
        t = thresholds.get(ch)
        if t is None or pd.isna(val):
            cats.append('large')
        elif val <= t[0]:
            cats.append('small')
        elif val <= t[1]:
            cats.append('mid')
        else:
            cats.append('large')
    return cats

def mean_distance_in_model(coordinates_stack, gene_bins, enh_bins):
    # ensemble-mean Euclidean 3D distance for paired bins (mirrors models.py:80-83)
    gene_bins = np.asarray(gene_bins, dtype=int)
    enh_bins = np.asarray(enh_bins, dtype=int)
    a = coordinates_stack[:, gene_bins, :]
    b = coordinates_stack[:, enh_bins, :]
    return np.linalg.norm(a - b, axis=2).mean(axis=0)

In [ ]:
def read_results(cell_line, query_id):
    return (spark.read.parquet(f"s3a://database/results/{query_id}")
            .withColumn("cell_line", F.lit(cell_line)))

results = None
for cl, qid in QUERY_IDS.items():
    df = read_results(cl, qid)
    results = df if results is None else results.union(df)

results = (results
           .where("avg_dist > 0 AND var_dist > 0")
           .where(F.col('project_id').isin(USED_PROJECTS)))

chromatin_states_df = (spark.read.parquet("s3a://database/chromatin_states")
                       .where(F.col('name').isin(ACTIVE_STATES)))
results.createOrReplaceTempView("results")
chromatin_states_df.createOrReplaceTempView("chromatin_states")

In [ ]:
active_pairs = spark.sql("""
SELECT r.gene_id, r.gene_chr, r.gene_start, r.gene_end, r.gene_strand,
       r.enh_id, r.enh_chr, r.enh_start, r.enh_end,
       r.avg_dist, r.cell_line
FROM results r
WHERE EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.gene_chr
                AND cs.start <= r.gene_end AND cs.end >= r.gene_start)
  AND EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.enh_chr
                AND cs.start <= r.enh_end AND cs.end >= r.enh_start)
""")

In [ ]:
w = Window.partitionBy('cell_line', 'gene_id').orderBy(F.col('avg_dist').asc())
nearest = (active_pairs
           .withColumn('rk', F.row_number().over(w))
           .where('rk = 1')
           .drop('rk'))

# per-gene tables are small (~17k genes/cell line) -> collect to the driver
nearest_pd = nearest.toPandas()
nearest_pd['gene_chr'] = nearest_pd['gene_chr'].astype(str)
print(nearest_pd.shape, nearest_pd.cell_line.value_counts().to_dict())
nearest_pd.head()

In [ ]:
proj = (spark.read.json("s3a://database/project_configuration", multiLine=True)
        .select(F.col('project_id'), F.explode('datasets').alias('d'))
        .select('project_id',
                F.col('d.ensemble_id').alias('ensemble_id'),
                F.col('d.ensemble_region.chromosome').alias('chrom'),
                F.col('d.metadata.cell_line').alias('cell_line'))
        .where(F.col('project_id').isin(USED_PROJECTS)))
proj_pd = proj.toPandas()
# (cell_line, chrom) -> ensemble_id (one full-chromosome ensemble each)
ENS_BY = {(r.cell_line, str(r.chrom)): r.ensemble_id for r in proj_pd.itertuples()}
print(len(ENS_BY), "ensembles mapped")

In [ ]:
fs = s3fs.S3FileSystem()   # adjust to the cluster fs config if needed
_ENS_CACHE = {}

def load_ensemble(ensemble_id):
    # -> (coords (n_models, n_bins, 3), first_bin, last_bin, resolution)
    # mirrors load_chromatin_model_ensemble_from_filesystem (packed.py:9)
    if ensemble_id in _ENS_CACHE:
        return _ENS_CACHE[ensemble_id]
    base = f"{MODEL_REPOSITORY_BUCKET}/{ensemble_id}"
    with fs.open(f"{base}.metadata.json", "r") as fh:
        meta = json.load(fh)
    with fs.open(f"{base}.coordinates.npy", "rb") as fh:
        coords = np.load(fh)
    out = (coords, int(meta['first_bin']), int(meta['last_bin']), int(meta['resolution']))
    _ENS_CACHE[ensemble_id] = out
    return out

In [ ]:
def dist_L1_in_c2(pair_df, c2):
    # 3D distance of each gene's c1-nearest enhancer (L1) measured in c2's model
    dists = np.full(len(pair_df), np.nan)
    oow = np.zeros(len(pair_df), dtype=bool)
    for chrom, idx in pair_df.groupby('gene_chr').groups.items():
        rows = pair_df.loc[idx]
        positions = pair_df.index.get_indexer(idx)
        ens_id = ENS_BY.get((c2, str(chrom)))
        if ens_id is None:
            oow[positions] = True
            continue
        coords, first_bin, last_bin, res = load_ensemble(ens_id)
        gene_tss = np.where(rows['gene_strand'].values == '+',
                            rows['gene_start_c1'].values, rows['gene_end_c1'].values)
        enh_ctr = (rows['enh_start_c1'].values + rows['enh_end_c1'].values) // 2
        in_win = ((gene_tss >= first_bin) & (gene_tss <= last_bin)
                  & (enh_ctr >= first_bin) & (enh_ctr <= last_bin))
        gbin = (gene_tss - first_bin) // res + 1
        ebin = (enh_ctr - first_bin) // res + 1
        if in_win.any():
            dists[positions[in_win]] = mean_distance_in_model(coords, gbin[in_win], ebin[in_win])
        oow[positions[~in_win]] = True
    return dists, oow

In [ ]:
def build_comparison(c1, c2):
    n1 = nearest_pd[nearest_pd.cell_line == c1].set_index('gene_id')
    n2 = nearest_pd[nearest_pd.cell_line == c2].set_index('gene_id')

    th_c1 = chrom_tertile_thresholds(n1.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')
    th_c2 = chrom_tertile_thresholds(n2.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')

    genes = n1.index.intersection(n2.index)
    df = pd.DataFrame({'gene_id': list(genes)})
    df['gene_chr'] = n1.loc[genes, 'gene_chr'].values
    df['gene_start_c1'] = n1.loc[genes, 'gene_start'].values
    df['gene_end_c1'] = n1.loc[genes, 'gene_end'].values
    df['gene_strand'] = n1.loc[genes, 'gene_strand'].values
    df['nearest_enh_c1'] = n1.loc[genes, 'enh_id'].values
    df['enh_start_c1'] = n1.loc[genes, 'enh_start'].values
    df['enh_end_c1'] = n1.loc[genes, 'enh_end'].values
    df['avg_dist_c1'] = n1.loc[genes, 'avg_dist'].values
    df['nearest_enh_c2'] = n2.loc[genes, 'enh_id'].values
    df['avg_dist_c2'] = n2.loc[genes, 'avg_dist'].values

    # proximity vs each cell line's own per-chromosome thresholds (decision 3)
    df['proximity_category_c1'] = proximity_categories(df['avg_dist_c1'], df['gene_chr'], th_c1)
    df['proximity_category_c2'] = proximity_categories(df['avg_dist_c2'], df['gene_chr'], th_c2)

    # Step 2: c1-nearest enhancer distance recomputed in c2's model,
    # classified against c2's thresholds (out-of-window / NaN -> large)
    d, oow = dist_L1_in_c2(df, c2)
    df['dist_L1_in_c2'] = d
    df['L1_out_of_c2_window'] = oow
    df['proximity_of_L1_in_c2'] = proximity_categories(df['dist_L1_in_c2'], df['gene_chr'], th_c2)

    # flags (decisions 4 & 5)
    df['nearest_changed'] = df['nearest_enh_c1'] != df['nearest_enh_c2']
    old_far = df['proximity_of_L1_in_c2'].eq('large')
    new_close = df['proximity_category_c2'].eq('small')
    df['is_redundancy'] = df['nearest_changed'] & old_far & new_close
    df['is_nonredundant_switcher'] = df['nearest_changed'] & old_far & ~new_close

    df.insert(0, 'comparison', f"{c1.lower()}_vs_{c2.lower()}")
    df.insert(1, 'c1', c1)
    df.insert(2, 'c2', c2)
    return df

In [ ]:
def redundancy_frequency(df):
    denom = int(df['nearest_changed'].sum())
    return float('nan') if denom == 0 else int(df['is_redundancy'].sum()) / denom

for c1, c2 in COMPARISONS:
    t = build_comparison(c1, c2)
    print(f"{c1}->{c2}: genes={len(t)}  redundancy={int(t.is_redundancy.sum())}  "
          f"frequency={redundancy_frequency(t):.4f}")
    out = f"s3a://database/enhancer_redundancy/{c1.lower()}_vs_{c2.lower()}"
    spark.createDataFrame(t).repartition(1).write.mode('overwrite').parquet(out)

print("done — pull s3a://database/enhancer_redundancy/* to "
      "data/whole_chromosomes/enhancer_redundancy/ for notebook 7")